In [2]:
import json
from pathlib import Path

# Load your raw file
path = list(Path("../data/raw/").glob("*.json"))[0]
with open(path) as f:
    data = json.load(f)

# What's the top-level structure?
print(type(data))
print(len(data))  # or data.keys() if it's a dict

# Look at one record
import pprint
pprint.pprint(data["jobs"][0])  # adjust if nested differently

<class 'dict'>
4
{'_geoloc': [{'lat': -22.9056, 'lon': -47.0626}],
 'apply_url': 'https://lde.tbe.taleo.net/lde01/ats/careers/requisition.jsp?org=BRP2&cws=52&rid=34937',
 'board_token': 'lde.tbe_lde01_BRP2_52',
 'collapse_key': '648b664640725e137b8475d87b26320c3add01de004c6b5204c17717e6cf693d',
 'enriched_company_data': {'activities': ['Designing powersports vehicles',
                                          'Manufacturing recreational '
                                          'watercraft',
                                          'Producing marine propulsion systems',
                                          'Developing internal combustion '
                                          'engines',
                                          'Manufacturing off-road vehicles'],
                           'enriched_at': '2026-03-12T02:20:24.500Z',
                           'homepage_uri': 'brp.com',
                           'hq_country': 'CA',
                           'industries': 

In [5]:
jobs = data['jobs']
print(f"Total jobs: {len(jobs)}")

populated = sum(1 for r in jobs if r.get('v5_processed_job_data', {}).get('technical_tools'))
print(f"{populated}/{len(jobs)} records have technical_tools")

with_salary = sum(1 for r in jobs if r.get('v5_processed_job_data', {}).get('yearly_max_compensation'))
print(f"{with_salary}/{len(jobs)} records have salary data")

Total jobs: 26143
26102/26143 records have technical_tools
9075/26143 records have salary data


## Data Quality Notes

- Records nested under data['jobs'] — 26,143 total
- Unique ID field: 'id' (use for deduplication)
- technical_tools: 99.8% populated — use this for skill extraction, skip HTML parsing
- salary (yearly_min/max_compensation): 34.7% populated — filter nulls in analysis
- job descriptions: raw HTML, needs stripping (beautifulsoup) — lower priority given technical_tools quality
- estimated_publish_date: ISO format, clean
- workplace_countries: use for NL filtering — check how many are actually NL
- _geoloc: present but not needed

In [6]:
nl_jobs = sum(1 for r in jobs if 'NL' in r.get('v5_processed_job_data', {}).get('workplace_countries', []))
print(f"NL jobs: {nl_jobs}/{len(jobs)}")

NL jobs: 286/26143
